# Trekomend v2 — Qwen3-Embedding-0.6B · 1024-dim · Kaggle Dual T4

Generates **1024-dimensional** movie embeddings for **~1M TMDB movies**
on Kaggle's free dual-T4 GPU setup. One session, one zip download.

---

## Dataset

Uses **[asaniczka/tmdb-movies-dataset-2023-930k-movies]**
(linked in kernel metadata). ~1M movies, updated daily, 24 columns.

## What Changed from v1

| v1 | v2 (this) | Reason |
|---|---|---|
| 768-dim (MRL trunc) | **1024-dim** (native) | +2-3% quality |
| TMDB id in text | **Removed** | Pure noise |
| Budget/revenue/ratings | **Removed** | Collaborative, not content |
| Flat field listing | **Layered priority** | Genres/keywords first |
| max_len=384 | **max_len=512** | More plot context |
| SDPA/eager fallback | **flash_attention_2 priority** | +20-25% speed |
| 20 fields in CSV | **12 fields** (content-only) | Less RAM, faster parse |

## Research-Backed Field Ranking

| Priority | Field | Impact |
|---|---|---|
| \ud83d\udd34 HIGH | overview (plot) | Richest semantic signal |
| \ud83d\udd34 HIGH | keywords | 2x more impactful than plot alone |
| \ud83d\udfe1 HIGH | genres | Essential coarse filter |
| \ud83d\udfe1 MED | title, year | Identity + era preference |
| \ud83d\udfe2 LOW | tagline, companies, countries, lang, runtime | Supplementary |
| \u274c NONE | id, budget, revenue, rating, popularity, adult | Noise |

---

## Hardware (Kaggle Free Tier)

| Resource | Spec |
|---|---|
| GPU | 2x NVIDIA Tesla T4 (16 GB VRAM each) |
| RAM | 29 GB |
| Disk | ~20 GB `/kaggle/working` (persisted) |
| Session | 12 hours max |
| Output HDF5 | ~3.8 GB (1024-dim, ~1M rows) |

## Quick Start

1. Kaggle  New Notebook  Settings  **GPU T4 x2**
2. Add Dataset: **asaniczka/tmdb-movies-dataset-2023-930k-movies**
3. **Runtime  Run All**
4. Last cell creates downloadable zip

**Pipeline: 25-50 minutes for ~1M movies.**

In [ ]:
import sys, subprocess

REQUIRED = [
    "transformers>=4.51.0", "accelerate", "sentencepiece",
    "safetensors", "tokenizers", "h5py", "tqdm", "psutil",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", *REQUIRED],
    check=True,
)

# Try flash-attn install (may fail on some Kaggle images  that's OK)
try:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "flash-attn", "--no-build-isolation"],
        check=True, timeout=300,
    )
    print("\u2713 flash-attn installed.")
except Exception:
    print("\u26a0 flash-attn not available  will use sdpa/eager fallback.")

import torch, transformers
print(f"Python {sys.version.split()[0]}  |  "
      f"PyTorch {torch.__version__}  |  "
      f"Transformers {transformers.__version__}")

assert transformers.__version__ >= "4.51.0", \
    "Need transformers >= 4.51.0 for Qwen3. Restart kernel & re-run."

print("\u2713 Dependencies OK.")

In [ ]:
import os, sys, gc, json, math, shutil, sqlite3, time, zipfile, warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path

import h5py, numpy as np, pandas as pd, psutil
import torch, torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

warnings.filterwarnings("ignore")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

MODEL_ID      = "Qwen/Qwen3-Embedding-0.6B"
OUT_DIM       = 1024            # Native dimension. 768/512/256 for smaller files.
MAX_LEN       = 512             # Tokens. 256=faster, 512=richer context, 1024=full.
MODEL_DTYPE   = torch.float16   # T4-optimized (no BF16 on T4; FP16 uses Tensor Cores)

TOTAL_BATCH_SIZE   = 448        # 224 per GPU. Slightly less than v1 (1024-dim output).
MIN_BATCH_SIZE     = 64         # Auto-reduce floor on OOM
SHARD_SIZE         = 10_000     # Rows per HDF5 shard
CPU_WORKERS        = 4          # Thread pool for text building

OVERVIEW_CHAR_LIMIT  = 2000     # Plot  richest content signal
KEYWORDS_CHAR_LIMIT  = 800      # Themes/topics  very high signal
GENRES_CHAR_LIMIT    = 500      # Genre list  high signal
TAGLINE_CHAR_LIMIT   = 200      # Low signal, keep short
COMPANIES_CHAR_LIMIT = 250      # Studio  low signal, condensed
COUNTRIES_CHAR_LIMIT = 150      # Country  low signal

SHARD_COMPRESSION   = "lzf"     # Fast temp compression
MERGED_COMPRESSION  = "gzip"    # Portable final file
MERGED_GZIP_LEVEL   = 2
H5_CHUNK_ROWS       = 512
DELETE_SHARDS_AFTER_MERGE = True  # Save disk space on Kaggle

LOG_EVERY_BATCHES     = 8
VALIDATION_CHUNK_ROWS = 25_000
RANDOM_SEED           = 42
WARMUP_BATCHES        = 8
TIMED_BATCHES         = 16
np.random.seed(RANDOM_SEED)

WORK_DIR     = Path("/kaggle/working")
TMP_DIR      = Path("/kaggle/tmp")
INPUT_DIR    = Path("/kaggle/input")
# Specific dataset path (asaniczka TMDB 2023 930k+ movies)
DATASET_PATH = INPUT_DIR / "tmdb-movies-dataset-2023-930k-movies"

OUTPUT_DIR   = WORK_DIR / "trekomend_v2_output"
SHARD_DIR    = TMP_DIR / "trekomend_shards"  # Scratch: doesn't count against output limit
DB_PATH      = OUTPUT_DIR / "checkpoint.db"
LOG_PATH     = OUTPUT_DIR / "run.log"
MANIFEST_PATH = OUTPUT_DIR / "manifest.json"
MERGED_H5    = WORK_DIR / "tmdb_qwen06b_1024d.h5"
ZIP_PATH     = WORK_DIR / "trekomend_v2_1024d.zip"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SHARD_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

CSV_USECOLS = [
    "id",                    # For index tracking only  NOT in embedding text
    "title",                 # Identity
    "original_title",        # Alternative name (only used if differs)
    "release_date",          #  year extraction
    "runtime",               # Length context
    "original_language",     # Language context
    "overview",              # \u2605 HIGHEST: plot summary
    "tagline",               # Supplementary
    "genres",                # \u2605 HIGH: genre classification
    "keywords",              # \u2605 VERY HIGH: themes/elements
    "production_companies",  # Studio context
    "production_countries",  # Country context
]

CSV_DTYPES = {
    "id": "int32",
    "runtime": "float32",
}

N_GPUS = torch.cuda.device_count()

print(f"{'='*60}")
print(f"Trekomend v2 (Kaggle Kernel)  {MODEL_ID}")
print(f"  dtype={MODEL_DTYPE}  dim={OUT_DIM}  max_len={MAX_LEN}")
print(f"  batch={TOTAL_BATCH_SIZE} total ({TOTAL_BATCH_SIZE//max(1,N_GPUS)}/GPU)")
print(f"  shard={SHARD_SIZE:,} rows  workers={CPU_WORKERS}")
print(f"  GPUs detected: {N_GPUS}")
print(f"  Dataset: asaniczka/tmdb-movies-dataset-2023-930k-movies")
print(f"{'='*60}")
if N_GPUS < 2:
    print(f"  \u26a0 WARNING: Need 2 GPUs. Settings  Accelerator  GPU T4 x2")

In [ ]:
PROCESS = psutil.Process(os.getpid())

def utc_now() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

def log(msg: str):
    line = f"[{utc_now()}] {msg}"
    try:
        tqdm.write(line)
    except Exception:
        print(line)
    try:
        with open(LOG_PATH, "a", encoding="utf-8") as f:
            f.write(line + "\n")
    except Exception:
        pass

def ram_gb() -> float:
    return PROCESS.memory_info().rss / 1e9

def disk_free_gb(path: Path = WORK_DIR) -> float:
    try:
        return shutil.disk_usage(path).free / 1e9
    except Exception:
        return -1.0

def disk_usage_gb(path: Path) -> float:
    """Total disk used by a directory tree."""
    try:
        total = 0
        for root, dirs, files in os.walk(path):
            for f in files:
                fp = os.path.join(root, f)
                try:
                    total += os.path.getsize(fp)
                except OSError:
                    pass
        return total / 1e9
    except Exception:
        return -1.0

def gpu_snapshot() -> list:
    if not torch.cuda.is_available():
        return []
    stats = []
    for i in range(torch.cuda.device_count()):
        free_b, total_b = torch.cuda.mem_get_info(i)
        stats.append({
            "id": i,
            "name": torch.cuda.get_device_name(i),
            "free_gb": free_b / 1e9,
            "total_gb": total_b / 1e9,
            "alloc_gb": torch.cuda.memory_allocated(i) / 1e9,
            "reserved_gb": torch.cuda.memory_reserved(i) / 1e9,
        })
    return stats

def gpu_short() -> str:
    stats = gpu_snapshot()
    if not stats:
        return "cpu"
    parts = []
    for s in stats:
        parts.append(
            f"GPU{s['id']} alloc={s['alloc_gb']:.1f}G "
            f"free={s['free_gb']:.1f}G"
        )
    return " | ".join(parts)

def clear_memory(empty_cuda: bool = True):
    gc.collect()
    if empty_cuda and torch.cuda.is_available():
        torch.cuda.empty_cache()

def print_memory_report(title: str = "memory"):
    log(f"{title}: RAM={ram_gb():.1f}G  "
        f"disk_free={disk_free_gb():.1f}G  GPU=[{gpu_short()}]")

# Write log header
with open(LOG_PATH, "a", encoding="utf-8") as f:
    f.write(f"\n{'='*70}\n")
    f.write(f"[{utc_now()}] Trekomend v2 (1024d) Kaggle Kernel session\n")
    f.write(f"  Dataset: asaniczka/tmdb-movies-dataset-2023-930k-movies\n")
    f.write(f"  Model: {MODEL_ID}  dim={OUT_DIM}  max_len={MAX_LEN}\n")
    f.write(f"{'='*70}\n")

print("Logger & GPU monitor ready.")

In [ ]:
CSV_PATH = None

# Priority 1: asaniczka dataset via kernel-metadata.json dataset_sources
if DATASET_PATH.exists():
    for f in DATASET_PATH.iterdir():
        if f.suffix == ".csv" and "TMDB_movie_dataset" in f.name:
            CSV_PATH = f
            log(f"Found dataset: asaniczka  {f.name}")
            break

# Priority 2: search /kaggle/input and /kaggle/working for any TMDB CSV
if CSV_PATH is None:
    for search_dir in [INPUT_DIR, WORK_DIR]:
        if not search_dir.exists():
            continue
        for root, dirs, files in os.walk(search_dir):
            for name in files:
                if name.endswith("TMDB_movie_dataset_v11.csv"):
                    CSV_PATH = Path(root) / name
                    log(f"Found CSV: {CSV_PATH}")
                    break
            if CSV_PATH:
                break
        if CSV_PATH:
            break

# Priority 3: HuggingFace download fallback
if CSV_PATH is None:
    CSV_PATH = WORK_DIR / "TMDB_movie_dataset_v11.csv"
    if not CSV_PATH.exists():
        log("Downloading TMDB dataset from HuggingFace (~632 MB)...")
        subprocess.run([
            "wget", "-q", "--show-progress", "-O", str(CSV_PATH),
            "https://huggingface.co/datasets/fukitweball/TMDB/resolve/main/TMDB_movie_dataset_v11.csv"
        ], check=True)

log(f"CSV path: {CSV_PATH}  ({CSV_PATH.stat().st_size/1e9:.2f} GB)")

log("Counting rows...")
with open(CSV_PATH, "r", encoding="utf-8") as f:
    N_ROWS = sum(1 for _ in f) - 1  # minus header
log(f"Total rows: {N_ROWS:,}")

n_shards = math.ceil(N_ROWS / SHARD_SIZE)
est_raw = N_ROWS * OUT_DIM * 4 / 1e9
est_gzip = est_raw * 0.55  # gzip level 2 roughly halves float32

print(f"\n{'='*60}")
print(f"DATASET:  {CSV_PATH.name}")
print(f"  Source:  asaniczka/tmdb-movies-dataset-2023-930k-movies")
print(f"  Rows:    {N_ROWS:,}")
print(f"  Columns in embedding text: {len(CSV_USECOLS)}")
print(f"  Est raw output:  {est_raw:.1f} GB ({OUT_DIM}-dim float32)")
print(f"  Est gzip output: ~{est_gzip:.1f} GB (final HDF5)")
print(f"  Shards:  {n_shards:,} \u00d7 {SHARD_SIZE:,} rows")
print(f"{'='*60}")

print_memory_report("starting")
for s in gpu_snapshot():
    print(f"  cuda:{s['id']}  {s['name']}: "
          f"{s['free_gb']:.1f}G free / {s['total_gb']:.1f}G")
print(f"  Disk: /kaggle/working free={disk_free_gb(WORK_DIR):.1f}G  "
      f"/kaggle/tmp free={disk_free_gb(TMP_DIR):.1f}G")
print()

In [ ]:
# DESIGN PRINCIPLES (backed by embedding survey ablation studies):
#   1. Overview (plot)  HIGHEST impact  richest semantic signal
#   2. Keywords  VERY HIGH  2\u00d7 more impactful than plot alone
#   3. Genres  HIGH  essential coarse clustering for recsys
#   4. Title + Year  MODERATE  identity + era preference
#   5. Production context  LOW  condensed, supplementary only
#   6. Budget/Revenue/Ratings/Popularity  REMOVED  collaborative not content
#   7. TMDB ID/IMDB ID/Poster paths/Homepage  REMOVED  pure noise
#
# TEXT LAYERING (fields ordered by semantic priority):
#   Layer 1: Title + Alternative Title (identity)
#   Layer 2: Genres, Keywords (strongest content signals)
#   Layer 3: Plot overview (richest text)
#   Layer 4: Tagline (supplementary)
#   Layer 5: Year, Runtime, Language, Country, Studio (context, condensed)

def safe_str(v, default: str = "") -> str:
    """Convert value to clean string, filtering NaN/None/empty."""
    if v is None:
        return default
    try:
        if isinstance(v, float) and (pd.isna(v) or not np.isfinite(v)):
            return default
    except Exception:
        pass
    s = str(v).strip()
    if not s or s.lower() in {"nan", "none", "null", ""}:
        return default
    return s

def clip_text(s: str, limit: int) -> str:
    """Truncate at word boundary. Adds '\u2026' if truncated."""
    if len(s) <= limit:
        return s
    cut = s[:limit].rstrip()
    # Try to break at last space
    last_space = cut.rfind(" ")
    if last_space > limit * 0.7:
        return cut[:last_space] + "\u2026"
    return cut + "\u2026"

def movie_text(row) -> str:
    """
    Build embedding text for one movie.
    Research-backed field ordering and limits.
    Only 12 content-relevant fields. No IDs, financials, or ratings.
    """
    title = safe_str(row.title, "Untitled")
    original_title = safe_str(row.original_title)
    release = safe_str(row.release_date)
    year = release[:4] if len(release) >= 4 and release[:4].isdigit() else ""

    overview = clip_text(safe_str(row.overview), OVERVIEW_CHAR_LIMIT)
    genres = clip_text(safe_str(row.genres), GENRES_CHAR_LIMIT)
    keywords = clip_text(safe_str(row.keywords), KEYWORDS_CHAR_LIMIT)
    tagline = clip_text(safe_str(row.tagline), TAGLINE_CHAR_LIMIT)
    companies = clip_text(safe_str(row.production_companies), COMPANIES_CHAR_LIMIT)
    countries = clip_text(safe_str(row.production_countries), COUNTRIES_CHAR_LIMIT)
    lang = safe_str(row.original_language)
    runtime = safe_str(row.runtime)

    #  Layer 1: Identity 
    lines = [f"Movie: {title}"]
    if original_title and original_title.lower() != title.lower():
        lines.append(f"Also known as: {original_title}")

    #  Layer 2: Strong semantic signals 
    if genres:
        lines.append(f"Genres: {genres}")
    if keywords:
        lines.append(f"Themes & elements: {keywords}")

    #  Layer 3: Rich content 
    if overview:
        lines.append(f"Plot: {overview}")

    #  Layer 4: Supplementary 
    if tagline:
        lines.append(f"Tagline: {tagline}")

    #  Layer 5: Production context (condensed single line) 
    meta = []
    if year:
        meta.append(f"Year: {year}")
    if runtime:
        meta.append(f"{runtime} min")
    if lang:
        meta.append(f"Language: {lang}")
    if countries:
        meta.append(f"Country: {countries}")
    if companies:
        meta.append(f"Studio: {companies}")
    if meta:
        lines.append(" | ".join(meta))

    # NOTE: No TMDB ID, IMDB ID, budget, revenue, rating, popularity,
    # adult flag, poster path, backdrop path, homepage, or spoken languages.
    # These are collaborative signals or noise  not content similarity.

    return "\n".join(lines)


def build_texts_parallel(rows, desc: str = "build text") -> list:
    """Build movie texts in parallel across CPU threads."""
    n = len(rows)
    texts = [None] * n
    with ThreadPoolExecutor(max_workers=CPU_WORKERS) as pool:
        future_to_idx = {pool.submit(movie_text, row): i for i, row in enumerate(rows)}
        with tqdm(total=n, desc=desc, unit="movie", leave=False, dynamic_ncols=True) as bar:
            for fut in as_completed(future_to_idx):
                idx = future_to_idx[fut]
                try:
                    texts[idx] = fut.result()
                except Exception as e:
                    texts[idx] = f"Error: {e}"
                bar.update(1)
    return texts

# Quick validation: print a sample text for visual inspection
sample_row = pd.Series({
    "title": "Inception", "original_title": "Inception",
    "release_date": "2010-07-16", "runtime": 148.0,
    "original_language": "en",
    "overview": "Cobb, a skilled thief who commits corporate espionage "
                "by infiltrating the subconscious of his targets...",
    "tagline": "Your mind is the scene of the crime.",
    "genres": "Action, Science Fiction, Adventure",
    "keywords": "dream, subconscious, heist, mind-bending, thriller",
    "production_companies": "Warner Bros. Pictures, Legendary Entertainment",
    "production_countries": "United States of America, United Kingdom",
})
print("\nSample text template output:")
print("-" * 50)
print(movie_text(sample_row))
print("-" * 50)
print(f"Template: identity  genres+keywords  plot  tagline  context")
print(f"Excluded: budget, revenue, ratings, popularity, adult, paths, IDs")

In [ ]:
# \n
# Cell 6  Checkpoint Database (SQLite, crash-resilient)
# \n
def init_db() -> sqlite3.Connection:
    con = sqlite3.connect(str(DB_PATH))
    con.execute("PRAGMA journal_mode=WAL")
    con.execute("PRAGMA synchronous=NORMAL")
    con.execute("PRAGMA busy_timeout=5000")
    con.execute("""
        CREATE TABLE IF NOT EXISTS shards (
            shard_idx   INTEGER PRIMARY KEY,
            first_row   INTEGER NOT NULL,
            last_row    INTEGER NOT NULL,
            n_rows      INTEGER NOT NULL,
            status      TEXT NOT NULL DEFAULT 'pending',
            file        TEXT,
            attempts    INTEGER NOT NULL DEFAULT 0,
            started_at  TEXT,
            finished_at TEXT,
            last_error  TEXT
        )
    """)
    con.commit()
    return con

def shard_status(con, shard_idx: int) -> str | None:
    row = con.execute(
        "SELECT status FROM shards WHERE shard_idx = ?", (shard_idx,)
    ).fetchone()
    return row[0] if row else None

def mark_shard_start(con, shard_idx: int, first_row: int, last_row: int, n_rows: int):
    con.execute(
        """INSERT OR REPLACE INTO shards (shard_idx, first_row, last_row, n_rows,
           status, started_at, attempts)
           VALUES (?, ?, ?, ?, 'running', ?,
           COALESCE((SELECT attempts FROM shards WHERE shard_idx=?), 0) + 1)""",
        (shard_idx, first_row, last_row, n_rows, utc_now(), shard_idx),
    )
    con.commit()

def mark_shard_done(con, shard_idx: int, h5_path: str):
    con.execute(
        "UPDATE shards SET status='done', file=?, finished_at=? WHERE shard_idx=?",
        (h5_path, utc_now(), shard_idx),
    )
    con.commit()

def mark_shard_error(con, shard_idx: int, error: str):
    con.execute(
        "UPDATE shards SET status='error', last_error=?, finished_at=? WHERE shard_idx=?",
        (error[:500], utc_now(), shard_idx),
    )
    con.commit()

print("\u2713 Checkpoint DB ready.")

In [ ]:
# \n
# Cell 7  Load Model  (Manual Dual GPU + Flash Attention + torch.compile)
# \n
#
# STRATEGY (research-backed for max throughput on dual T4):
#   1. Load two independent model instances (one per GPU)
#   2. Manual thread dispatch: Thread A  GPU0, Thread B  GPU1
#   3. Both GPUs tokenize+forward CONCURRENTLY (Rust tokenizer releases GIL)
#   4. Flash Attention 2 priority (T4 supports FA2, +20-25% speed)
#   5. torch.compile attempt (PyTorch 2.x, +4-40% speed on repeated forwards)
#
# WHY NOT DataParallel? Manual replicas yield better throughput.
# DataParallel re-replicates and gathers every forward  overhead for small batches.
# Manual replicas: zero per-batch overhead, both GPUs work entirely concurrently.
# \n

log(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, padding_side="left", trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
log(f"Tokenizer ready. vocab={tokenizer.vocab_size}")

#  Attention: flash_attention_2  sdpa  eager 
ATTN_ORDER = ["flash_attention_2", "sdpa", "eager"]
load_kwargs = dict(
    torch_dtype=MODEL_DTYPE,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

attn_used = None
model_kwargs = None

for attn in ATTN_ORDER:
    try:
        log(f"Testing attn_implementation='{attn}'...")
        test_model = AutoModel.from_pretrained(
            MODEL_ID, attn_implementation=attn, **load_kwargs,
        )
        del test_model
        clear_memory()
        attn_used = attn
        model_kwargs = {**load_kwargs, "attn_implementation": attn}
        log(f"\u2713 Using attn_implementation='{attn}'")
        break
    except Exception as e:
        log(f"  '{attn}' failed: {str(e)[:120]}")
        clear_memory()

if model_kwargs is None:
    attn_used = "default"
    model_kwargs = dict(load_kwargs)
    log("Falling back to default attention.")

#  Load two independent instances (one per GPU) 
log("Loading model on GPU 0...")
model_0 = AutoModel.from_pretrained(MODEL_ID, **model_kwargs)
model_0 = model_0.to("cuda:0")
model_0.eval()

log("Loading model on GPU 1...")
model_1 = AutoModel.from_pretrained(MODEL_ID, **model_kwargs)
model_1 = model_1.to("cuda:1")
model_1.eval()

params_m = sum(p.numel() for p in model_0.parameters()) / 1e6
log(f"Models loaded: {params_m:.0f}M params each, attn='{attn_used}'")

#  torch.compile (PyTorch 2.x)  extra 4-40% for repeated forwards \n
compile_used = False
if hasattr(torch, "compile") and torch.__version__ >= "2.0":
    try:
        log("Attempting torch.compile (mode='reduce-overhead')...")
        model_0 = torch.compile(model_0, mode="reduce-overhead")
        model_1 = torch.compile(model_1, mode="reduce-overhead")
        compile_used = True
        log("\u2713 torch.compile enabled on both GPUs")
    except Exception as e:
        log(f"  torch.compile failed: {str(e)[:120]} (using eager)")
        # Reload uncompiled models on failure
        del model_0, model_1
        clear_memory()
        model_0 = AutoModel.from_pretrained(MODEL_ID, **model_kwargs).to("cuda:0").eval()
        model_1 = AutoModel.from_pretrained(MODEL_ID, **model_kwargs).to("cuda:1").eval()

#  Warmup both GPUs (prime CUDA, trigger compile) 
log("GPU warmup (priming CUDA graphs + compile caches)...")
warm_texts = ["Warmup: test embedding initialization."] * 16
enc = tokenizer(warm_texts, padding=True, truncation=True,
                max_length=MAX_LEN, return_tensors="pt")

with torch.inference_mode():
    _ = model_0(input_ids=enc["input_ids"].to("cuda:0", non_blocking=True),
                attention_mask=enc["attention_mask"].to("cuda:0", non_blocking=True))
    _ = model_1(input_ids=enc["input_ids"].to("cuda:1", non_blocking=True),
                attention_mask=enc["attention_mask"].to("cuda:1", non_blocking=True))
torch.cuda.synchronize()
del enc, _
clear_memory()

#  Verify GPU memory 
log("Post-load GPU state:")
for s in gpu_snapshot():
    log(f"  cuda:{s['id']}  alloc={s['alloc_gb']:.2f}G  "
        f"reserved={s['reserved_gb']:.2f}G  free={s['free_gb']:.2f}G")

print_memory_report("after model load")
print(f"\nAttn: {attn_used}  |  compile: {compile_used}  |  "
      f"Models: 2\u00d7{params_m:.0f}M  |  GPUs: {N_GPUS}")

In [ ]:
# \n
# Cell 8  Embedding Engine (Dual GPU, Last Token Pool, MRL, Concurrent Forward)
# \n

def last_token_pool(hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """
    Qwen3 embedding: extract last non-padding token, then normalize.
    Matches official Qwen3-Embedding implementation exactly.
    With left-padding, the last position IS the last token.
    """
    left = (mask[:, -1].sum() == mask.shape[0])
    if left:
        return hidden[:, -1, :]
    lengths = mask.sum(dim=1) - 1
    return hidden[torch.arange(hidden.shape[0], device=hidden.device), lengths]


@torch.inference_mode()
def _encode_gpu_tensor(texts: list, gpu_id: int,
                       model: torch.nn.Module) -> torch.Tensor:
    """
    Encode texts on a specific GPU. Returns GPU tensor (NO CPU sync).

    This is CRITICAL for dual-GPU concurrency: both GPUs launch their
    forward passes concurrently via threads, then we sync BOTH to CPU together.
    """
    device = torch.device(f"cuda:{gpu_id}")
    enc = tokenizer(
        texts, padding=True, truncation=True,
        max_length=MAX_LEN, return_tensors="pt",
    )
    ids = enc["input_ids"].to(device, non_blocking=True)
    mask = enc["attention_mask"].to(device, non_blocking=True)
    del enc

    with torch.autocast(device_type="cuda", dtype=MODEL_DTYPE, enabled=True):
        outputs = model(input_ids=ids, attention_mask=mask)

    hidden = outputs[0] if isinstance(outputs, tuple) else outputs.last_hidden_state
    pooled = last_token_pool(hidden, mask)

    # Matryoshka: truncate to desired dimension (MRL preserves quality)
    pooled = pooled[:, :OUT_DIM]
    pooled = F.normalize(pooled.float(), p=2, dim=1)

    del ids, mask, outputs, hidden
    return pooled  # GPU tensor  caller syncs to CPU


def embed_batch(texts: list) -> np.ndarray:
    """
    Encode a batch using BOTH GPUS CONCURRENTLY via threads.

    Thread A  GPU0 forward  returns GPU tensor
    Thread B  GPU1 forward  returns GPU tensor
    Both tokenize concurrently (Rust tokenizer releases GIL).
    Both GPUs compute concurrently. Then both sync to CPU.
    """
    n = len(texts)
    if N_GPUS >= 2 and n >= 4:
        mid = n // 2
        with ThreadPoolExecutor(max_workers=2) as pool:
            fut0 = pool.submit(_encode_gpu_tensor, texts[:mid], 0, model_0)
            fut1 = pool.submit(_encode_gpu_tensor, texts[mid:], 1, model_1)
            gpu0_tensor = fut0.result()
            gpu1_tensor = fut1.result()

        a = gpu0_tensor.cpu().numpy().astype(np.float32, copy=False)
        b = gpu1_tensor.cpu().numpy().astype(np.float32, copy=False)
        del gpu0_tensor, gpu1_tensor
        return np.concatenate([a, b], axis=0)
    else:
        # Single GPU fallback
        t = _encode_gpu_tensor(texts, 0, model_0)
        result = t.cpu().numpy().astype(np.float32, copy=False)
        del t
        return result


def is_cuda_oom(exc: Exception) -> bool:
    msg = str(exc).lower()
    return "out of memory" in msg or isinstance(exc, torch.cuda.OutOfMemoryError)


def embed_texts_to_array(texts: list, shard_idx: int,
                         start_bs: int = TOTAL_BATCH_SIZE) -> tuple:
    """
    Encode a full shard with adaptive batching + length-sorting.

    - Sorts by char length  less padding waste in batches
    - Pre-allocates output array (no list+stack overhead)
    - Adaptive OOM: halves batch on error, gently grows back
    """
    n = len(texts)
    out = np.empty((n, OUT_DIM), dtype=np.float32)

    # Sort by text length for minimal padding  faster encoding
    order = sorted(range(n), key=lambda i: len(texts[i]))
    sorted_texts = [texts[i] for i in order]

    bs = start_bs
    good_batches = 0
    i = 0

    with tqdm(total=n, desc=f"shard {shard_idx} GPU", unit="movie",
              leave=False, dynamic_ncols=True) as bar:
        while i < n:
            end = min(i + bs, n)
            batch = sorted_texts[i:end]
            batch_indices = order[i:end]

            try:
                vecs = embed_batch(batch)
                out[batch_indices, :] = vecs
                i = end
                good_batches += 1
                bar.update(len(batch))

                if good_batches % LOG_EVERY_BATCHES == 0 or i >= n:
                    bar.set_postfix(bs=bs, RAM=f"{ram_gb():.1f}G", GPU=gpu_short())

                # Gradually grow batch back after OOM recovery
                if good_batches % 8 == 0 and bs < start_bs:
                    bs = min(start_bs, bs + 48)

                del vecs

            except Exception as e:
                if is_cuda_oom(e) and bs > MIN_BATCH_SIZE:
                    old_bs = bs
                    bs = max(MIN_BATCH_SIZE, bs // 2)
                    log(f"CUDA OOM shard {shard_idx}: batch {old_bs}{bs}")
                    clear_memory()
                    continue
                raise

    # Quick quality check
    if not np.isfinite(out).all():
        bad = (~np.isfinite(out)).sum()
        raise RuntimeError(f"{bad} NaN/Inf values in shard output")

    norms = np.linalg.norm(out[: min(256, n)], axis=1)
    if abs(float(norms.mean()) - 1.0) > 0.05:
        raise RuntimeError(f"Norm mean {norms.mean():.4f} != 1.0")

    return out, bs


print(f"\u2713 Embedding engine ready.")
print(f"  Strategy: manual dual-GPU ({N_GPUS} instances) + concurrent threads")
print(f"  Batch: {TOTAL_BATCH_SIZE} total ({TOTAL_BATCH_SIZE//max(1,N_GPUS)} per GPU)")
print(f"  Pooling: last-token  Matryoshka {OUT_DIM}d  L2-norm")
print(f"  Dtype: {MODEL_DTYPE}  |  Attn: {attn_used}  |  Compile: {compile_used}")

In [ ]:
# \n
# Cell 9  Speed Benchmark
# \n
print("Running speed benchmark...\n")

# Generate realistic-length test texts (~300 chars avg, like actual movies)
test_texts = [
    "Movie: Test " + str(i) + ". Genres: Action, Sci-Fi, Adventure. "
    "Themes & elements: space travel, artificial intelligence, hero journey, "
    "dystopian future, moral dilemma. "
    "Plot: A test movie about embedding benchmarks and GPU throughput "
    "optimization on Kaggle notebooks. The protagonist must optimize "
    "dual GPU parallelism while navigating the complexities of CUDA. " * 2
    for i in range(TOTAL_BATCH_SIZE)
]

# Warmup (prime CUDA, trigger compile if enabled)
log(f"Warmup: {WARMUP_BATCHES} batches...")
for b in range(WARMUP_BATCHES):
    _ = embed_batch(test_texts)
torch.cuda.synchronize()
clear_memory()

# Timed run
log(f"Timed: {TIMED_BATCHES} batches...")
t0 = time.perf_counter()
for _ in range(TIMED_BATCHES):
    _ = embed_batch(test_texts)
torch.cuda.synchronize()
dt = time.perf_counter() - t0

rows_per_batch = TOTAL_BATCH_SIZE * TIMED_BATCHES
rows_per_sec = rows_per_batch / dt
ms_per_batch = (dt / TIMED_BATCHES) * 1000

shards_total = math.ceil(N_ROWS / SHARD_SIZE)
est_seconds = N_ROWS / rows_per_sec if rows_per_sec > 0 else float("inf")
est_minutes = est_seconds / 60

print(f"\n{'='*55}")
print(f"BENCHMARK RESULTS")
print(f"  Attention:      {attn_used}")
print(f"  torch.compile:  {compile_used}")
print(f"  Batch size:     {TOTAL_BATCH_SIZE} ({TOTAL_BATCH_SIZE//max(1,N_GPUS)}/GPU)")
print(f"  Speed:          {rows_per_sec:.0f} rows/s")
print(f"  Per batch:      {ms_per_batch:.0f} ms")
print(f"  Est total:      {est_minutes:.0f} min for {N_ROWS:,} rows")
print(f"  Est shard/sec:  {rows_per_sec/SHARD_SIZE:.2f} (shard={SHARD_SIZE:,})")
print(f"{'='*55}")

# Store for later reporting
BENCHMARK_RPS = rows_per_sec
BENCHMARK_MS = ms_per_batch

print_memory_report("after benchmark")

In [ ]:
# \n
# Cell 10  Main Processing Loop
# \n

def write_shard_atomic(h5_path: Path, first_row: int, last_row: int,
                       ids: np.ndarray, embeddings: np.ndarray):
    """Write shard to temp file then atomically rename  crash-safe."""
    tmp_path = h5_path.with_suffix(".tmp.h5")
    if tmp_path.exists():
        tmp_path.unlink()
    n = embeddings.shape[0]
    chunk_rows = min(H5_CHUNK_ROWS, n)
    with h5py.File(tmp_path, "w") as f:
        ek = dict(dtype="float32", chunks=(chunk_rows, OUT_DIM))
        if SHARD_COMPRESSION:
            ek["compression"] = SHARD_COMPRESSION
        f.create_dataset("embeddings", data=embeddings, **ek)
        f.create_dataset("ids", data=ids.astype(np.int32), dtype="int32",
                         chunks=(min(chunk_rows, len(ids)),))
        f.create_dataset("rows", data=np.arange(first_row, last_row, dtype=np.int32),
                         dtype="int32")
    tmp_path.replace(h5_path)


#  Initialize checkpoint DB \n
con = init_db()
total_shards = math.ceil(N_ROWS / SHARD_SIZE)

# Pre-populate shard entries
for si in range(total_shards):
    fr = si * SHARD_SIZE
    lr = min(fr + SHARD_SIZE, N_ROWS)
    nr = lr - fr
    if shard_status(con, si) is None:
        con.execute(
            "INSERT OR IGNORE INTO shards (shard_idx, first_row, last_row, n_rows) "
            "VALUES (?, ?, ?, ?)",
            (si, fr, lr, nr),
        )
con.commit()

already_done = con.execute(
    "SELECT COUNT(*) FROM shards WHERE status='done'"
).fetchone()[0]
print(f"Checkpoint: {total_shards} shards. "
      f"Already done: {already_done}")

#  Open CSV (streaming, content-only columns) \n
t_load = time.perf_counter()

reader = pd.read_csv(
    CSV_PATH,
    usecols=CSV_USECOLS,
    dtype=CSV_DTYPES,
    chunksize=SHARD_SIZE,
    low_memory=False,
)
log(f"CSV reader ready. {N_ROWS:,} rows, {len(CSV_USECOLS)} cols.")

#  Process each shard 
start_time = time.perf_counter()
rows_processed = 0
disc_errors = 0

for shard_idx, chunk in enumerate(reader):
    first_row = shard_idx * SHARD_SIZE
    last_row = first_row + len(chunk)
    n_chunk = len(chunk)

    # Skip already done shards (crash recovery)
    status = shard_status(con, shard_idx)
    if status == "done":
        rows_processed += n_chunk
        log(f"Shard {shard_idx}/{total_shards}: already done, skipping.")
        continue

    h5_path = SHARD_DIR / f"shard_{shard_idx:04d}.h5"

    log(f"\n{''*50}")
    log(f"Shard {shard_idx}/{total_shards}: rows {first_row:,}\u2013{last_row:,} ({n_chunk:,})")
    print_memory_report("start")

    mark_shard_start(con, shard_idx, first_row, last_row, n_chunk)

    try:
        # Step 1: Build texts (CPU parallel, 4 threads)
        t_text = time.perf_counter()
        texts = build_texts_parallel(
            list(chunk.itertuples()),
            desc=f"shard {shard_idx} text"
        )
        dt_text = time.perf_counter() - t_text
        log(f"  Text build: {dt_text:.1f}s ({n_chunk/dt_text:.0f} rows/s)")

        # Step 2: Embed (dual GPU, adaptive batching)
        t_emb = time.perf_counter()
        embeddings, final_bs = embed_texts_to_array(texts, shard_idx)
        dt_emb = time.perf_counter() - t_emb
        emb_rps = n_chunk / dt_emb if dt_emb > 0 else 0
        log(f"  Embedding:  {dt_emb:.1f}s ({emb_rps:.0f} rows/s, batch={final_bs})")

        # Step 3: Save (atomic write)
        t_save = time.perf_counter()
        ids = chunk["id"].values
        write_shard_atomic(h5_path, first_row, last_row, ids, embeddings)
        dt_save = time.perf_counter() - t_save
        shard_mb = h5_path.stat().st_size / 1e6
        log(f"  Save:       {dt_save:.1f}s ({shard_mb:.0f} MB)")

        mark_shard_done(con, shard_idx, str(h5_path))
        rows_processed += n_chunk

        # Progress
        elapsed = time.perf_counter() - start_time
        rps = rows_processed / elapsed if elapsed > 0 else 0
        eta = (N_ROWS - rows_processed) / rps if rps > 0 else 0
        log(f"  Progress: {rows_processed:,}/{N_ROWS:,} "
            f"({100*rows_processed/N_ROWS:.1f}%)  "
            f"avg {rps:.0f} r/s  ETA: {eta/60:.0f} min")

        # Disk check (warn if running low)
        df_w = disk_free_gb(WORK_DIR)
        if df_w < 2.0:
            log(f"  \u26a0 WARNING: Only {df_w:.1f}G free on /kaggle/working")

        # Cleanup
        del texts, embeddings, ids, chunk
        clear_memory()

    except Exception as e:
        log(f"  ERROR on shard {shard_idx}: {e}")
        mark_shard_error(con, shard_idx, str(e))
        disc_errors += 1
        clear_memory()
        if disc_errors > 3:
            log(f"  Too many errors ({disc_errors}). Aborting.")
            raise
        log(f"  Continuing to next shard...")
        continue

con.close()
elapsed_total = time.perf_counter() - start_time
avg_rps = rows_processed / elapsed_total if elapsed_total > 0 else 0

log(f"\n{'='*55}")
log(f"PROCESSING DONE")
log(f"  Rows:       {rows_processed:,}/{N_ROWS:,}")
log(f"  Time:       {elapsed_total/60:.1f} min")
log(f"  Speed:      {avg_rps:.0f} rows/s (benchmark was {BENCHMARK_RPS:.0f})")
log(f"  Efficiency: {avg_rps/BENCHMARK_RPS*100:.1f}% of benchmark")
log(f"  Errors:     {disc_errors}")
log(f"{'='*55}")
print_memory_report("final")

In [ ]:
# \n
# Cell 11  Merge Shards  Single HDF5 + Write Manifest
# \n
shard_files = sorted(SHARD_DIR.glob("shard_*.h5"))

if not shard_files:
    print("\u26a0 ERROR: No shard files found. Run Cell 10 first.")
else:
    rows_total = 0
    for sf in shard_files:
        with h5py.File(sf, "r") as f:
            rows_total += f["embeddings"].shape[0]

    log(f"Merging {len(shard_files)} shards  {rows_total:,} rows")
    print(f"Shards:   {len(shard_files)}")
    print(f"Rows:     {rows_total:,}")
    print(f"Dim:      {OUT_DIM}")
    print(f"Raw size: {rows_total*OUT_DIM*4/1e9:.1f} GB")

    if MERGED_H5.exists():
        MERGED_H5.unlink()

    chunk_rows = min(H5_CHUNK_ROWS, max(1, rows_total))

    with h5py.File(MERGED_H5, "w") as out_f:
        ek = dict(
            shape=(rows_total, OUT_DIM), dtype="float32",
            chunks=(chunk_rows, OUT_DIM),
            compression=MERGED_COMPRESSION,
        )
        if MERGED_COMPRESSION:
            ek["compression_opts"] = MERGED_GZIP_LEVEL
        emb_ds = out_f.create_dataset("embeddings", **ek)

        ids_ds = out_f.create_dataset(
            "ids", shape=(rows_total,), dtype="int32",
            chunks=(chunk_rows,),
        )

        # Stream merge: one shard at a time, never loads full dataset
        offset = 0
        with tqdm(total=rows_total, desc="merge", unit="row",
                  dynamic_ncols=True) as bar:
            for sf in shard_files:
                with h5py.File(sf, "r") as in_f:
                    n = in_f["embeddings"].shape[0]
                    emb_ds[offset:offset+n] = in_f["embeddings"][:]
                    ids_ds[offset:offset+n] = in_f["ids"][:]
                    offset += n
                    bar.update(n)

        # Metadata attributes
        out_f.attrs["model"] = MODEL_ID
        out_f.attrs["dim"] = OUT_DIM
        out_f.attrs["norm"] = "L2"
        out_f.attrs["pooling"] = "last_token"
        out_f.attrs["max_len"] = MAX_LEN
        out_f.attrs["attn"] = attn_used
        out_f.attrs["compile"] = compile_used
        out_f.attrs["dtype"] = str(MODEL_DTYPE)
        out_f.attrs["created"] = utc_now()
        out_f.attrs["n_rows"] = rows_total
        out_f.attrs["dataset"] = "asaniczka/tmdb-movies-dataset-2023-930k-movies"

    merged_gb = MERGED_H5.stat().st_size / 1e9
    log(f"Merged: {MERGED_H5} ({merged_gb:.2f} GB)")
    print(f"\n\u2713 Merged HDF5: {MERGED_H5}")
    print(f"  Size: {merged_gb:.2f} GB")
    print(f"  Rows: {rows_total:,} \u00d7 {OUT_DIM}")

    #  Write manifest.json (reproducibility) \n
    manifest = {
        "version": "trekomend_v2_1024d",
        "created": utc_now(),
        "model": {
            "id": MODEL_ID,
            "dim": OUT_DIM,
            "max_len": MAX_LEN,
            "dtype": str(MODEL_DTYPE),
            "attn": attn_used,
            "compile": compile_used,
            "pooling": "last_token",
            "norm": "L2",
        },
        "dataset": {
            "source": "asaniczka/tmdb-movies-dataset-2023-930k-movies",
            "file": CSV_PATH.name,
            "n_rows": N_ROWS,
        },
        "processing": {
            "n_shards": len(shard_files),
            "shard_size": SHARD_SIZE,
            "batch_size": TOTAL_BATCH_SIZE,
            "n_gpus": N_GPUS,
            "gpu_type": [s["name"] for s in gpu_snapshot()],
            "benchmark_rps": float(BENCHMARK_RPS),
            "avg_rps": float(avg_rps),
            "elapsed_min": float(elapsed_total / 60),
        },
        "template": {
            "fields_used": CSV_USECOLS,
            "fields_excluded": ["budget", "revenue", "vote_average", "vote_count",
                               "popularity", "status", "adult", "backdrop_path",
                               "poster_path", "homepage", "imdb_id", "spoken_languages"],
            "layer_order": ["title", "genres", "keywords", "overview", "tagline", "context"],
        },
        "output": {
            "file": MERGED_H5.name,
            "size_gb": round(merged_gb, 3),
            "n_rows": rows_total,
            "dim": OUT_DIM,
        },
    }
    with open(MANIFEST_PATH, "w") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    print(f"  Manifest: {MANIFEST_PATH}")

    #  Optionally delete shards to save disk \n
    if DELETE_SHARDS_AFTER_MERGE:
        log("Deleting shards to free disk space...")
        freed = 0
        for sf in shard_files:
            freed += sf.stat().st_size
            sf.unlink()
        log(f"Freed {freed/1e9:.2f} GB ({len(shard_files)} files)")
        print(f"  Freed {freed/1e9:.2f} GB by deleting shards")

    print_memory_report("after merge")

In [ ]:
# \n
# Cell 12  Validation
# \n
if not MERGED_H5.exists():
    print("\u26a0 ERROR: Merged file not found. Run Cell 11 first.")
else:
    print(f"\n{'='*60}")
    print(f"VALIDATION: {MERGED_H5}")
    print(f"{'='*60}")

    with h5py.File(MERGED_H5, "r") as f:
        emb = f["embeddings"]
        ids = f["ids"]
        n, d = emb.shape
        print(f"Shape:     {n:,} \u00d7 {d}")
        print(f"Model:     {f.attrs.get('model', '?')}")
        print(f"Attn:      {f.attrs.get('attn', '?')}")
        print(f"Compile:   {f.attrs.get('compile', '?')}")
        print(f"Dataset:   {f.attrs.get('dataset', '?')}")
        print(f"Size:      {MERGED_H5.stat().st_size/1e9:.2f} GB")

        # NaN/Inf scan
        bad, nsum, nssum, nmin, nmax, seen = 0, 0.0, 0.0, float("inf"), float("-inf"), 0
        with tqdm(total=n, desc="scan", unit="row", dynamic_ncols=True) as bar:
            for start in range(0, n, VALIDATION_CHUNK_ROWS):
                end = min(start + VALIDATION_CHUNK_ROWS, n)
                arr = emb[start:end]

                # NaN check
                nan_mask = ~np.isfinite(arr)
                bad += nan_mask.any(axis=1).sum()

                # Norm stats (on finite rows only)
                finite = arr[~nan_mask.any(axis=1)]
                if len(finite) > 0:
                    norms = np.linalg.norm(finite, axis=1)
                    nsum += norms.sum()
                    nssum += (norms ** 2).sum()
                    nmin = min(nmin, norms.min())
                    nmax = max(nmax, norms.max())
                    seen += len(finite)
                bar.update(end - start)

        print(f"\nNorm check ({seen:,} finite rows):")
        print(f"  NaN/Inf rows: {bad} ({100*bad/n:.4f}%)")
        mean_norm = nsum / seen if seen else 0
        std_norm = math.sqrt(max(0, nssum/seen - mean_norm**2)) if seen else 0
        print(f"  Mean norm:    {mean_norm:.6f}  (ideal: 1.0)")
        print(f"  Std norm:     {std_norm:.6f}")
        print(f"  Norm range:   [{nmin:.6f}, {nmax:.6f}]")

        # Pairwise similarity sample
        sample_size = min(2000, n)
        indices = np.random.choice(n, size=sample_size, replace=False)
        indices = np.sort(indices)
        sample = emb[indices]
        sim = sample @ sample.T
        np.fill_diagonal(sim, -9)
        print(f"\nPairwise similarity ({sample_size}\u00d7{sample_size} sample):")
        print(f"  Mean:  {sim.mean():.4f}")
        print(f"  Std:   {sim.std():.4f}")
        print(f"  Max:   {sim.max():.4f}")

        # Health checks
        if sim.max() > 0.999:
            print(f"  \u26a0 WARNING: Near-identical vectors  model collapse?")
        if abs(mean_norm - 1.0) > 0.01:
            print(f"  \u26a0 WARNING: Norms not unit  normalization may have failed")
        if bad > 0:
            print(f"  \u26a0 WARNING: {bad} NaN/Inf rows  corruption detected")
        if sim.max() < 0.2:
            print(f"  \u26a0 WARNING: Very low max similarity  embeddings may be random")

    print(f"\n{'='*60}")
    print("\u2713 Validation complete.")

In [ ]:
# \n
# Cell 13  Create Downloadable Zip
# \n
if not MERGED_H5.exists():
    print("\u26a0 ERROR: Merged file not found. Run Cell 11 first.")
else:
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()

    log(f"Creating zip: {ZIP_PATH}")
    print(f"Creating: {ZIP_PATH}")

    to_zip = [fp for fp in [MERGED_H5, MANIFEST_PATH, LOG_PATH] if fp.exists()]
    total_size = sum(fp.stat().st_size for fp in to_zip)
    print(f"Compressing {total_size/1e9:.2f} GB...")

    with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_STORED) as zf:
        for fp in to_zip:
            mb = fp.stat().st_size / 1e6
            print(f"  + {fp.name}  ({mb:.0f} MB)")
            zf.write(fp, arcname=fp.name)

    zip_gb = ZIP_PATH.stat().st_size / 1e9
    log(f"Zip ready: {ZIP_PATH} ({zip_gb:.2f} GB)")
    print(f"\n\u2713 ZIP: {ZIP_PATH} ({zip_gb:.2f} GB)")

    try:
        from IPython.display import FileLink, display
        display(FileLink(str(ZIP_PATH)))
        print("\nClick the link above to download, or find it in the Output tab.")
    except Exception:
        print(f"\nDownload from Kaggle Output tab: {ZIP_PATH}")

In [ ]:
# \n
# Cell 14  Semantic Smoke Test (Optional)
# \n
#
# This cell checks that the embeddings actually encode semantic similarity.
# It looks up a known movie and returns its nearest neighbors.
# If horror clusters with horror and sci-fi with sci-fi, the embeddings work.
#
QUERY_TITLE = "Interstellar"
TOP_K = 10

if not MERGED_H5.exists():
    print("Merged HDF5 not found. Run Cell 11 first.")
elif not CSV_PATH or not CSV_PATH.exists():
    print("CSV not found. Skipping smoke test.")
else:
    print("Loading title index...")
    meta = pd.read_csv(CSV_PATH, usecols=["id", "title", "genres", "overview"],
                       low_memory=False, dtype={"id": "int32"})

    # Find the query movie
    matches = meta[meta["title"].str.lower() == QUERY_TITLE.lower()]
    if matches.empty:
        matches = meta[meta["title"].str.contains(QUERY_TITLE, case=False, na=False)]

    if matches.empty:
        print(f"'{QUERY_TITLE}' not found. Sample titles: "
              f"{meta['title'].dropna().sample(5).tolist()}")
    else:
        q_id = int(matches.iloc[0]["id"])
        q_title = str(matches.iloc[0]["title"])
        q_genres = str(matches.iloc[0].get("genres", ""))

        id_to_title = dict(zip(meta["id"].astype(int), meta["title"].astype(str)))
        id_to_genres = dict(zip(meta["id"].astype(int), meta["genres"].fillna("")))
        del meta, matches
        clear_memory(empty_cuda=False)

        with h5py.File(MERGED_H5, "r") as f:
            all_ids = f["ids"][:]
            q_positions = np.where(all_ids == q_id)[0]

            if len(q_positions) == 0:
                print(f"ID {q_id} not in embeddings.")
            else:
                q_pos = int(q_positions[0])
                q_vec = f["embeddings"][q_pos]
                n_total = f["embeddings"].shape[0]

                # Cosine similarity via dot product (vectors are L2-normalized)
                scores = np.empty(n_total, dtype=np.float32)
                with tqdm(total=n_total, desc=f"search", unit="row",
                          dynamic_ncols=True) as bar:
                    for start in range(0, n_total, VALIDATION_CHUNK_ROWS):
                        end = min(start + VALIDATION_CHUNK_ROWS, n_total)
                        scores[start:end] = f["embeddings"][start:end] @ q_vec
                        bar.update(end - start)

                # Exclude self-match
                scores[q_pos] = -2.0
                top = np.argsort(-scores)[:TOP_K]

                print(f"\n Top {TOP_K} similar to: {q_title}  [{q_genres}]\n")
                print(f"{'Rank':<6} {'Score':<9} {'Title':<50} Genres")
                print("-" * 100)
                for rank, pos in enumerate(top, 1):
                    mid = int(all_ids[pos])
                    t = id_to_title.get(mid, str(mid))
                    g = id_to_genres.get(mid, "")
                    # Truncate long titles
                    if len(t) > 48:
                        t = t[:45] + "..."
                    print(f"{rank:<6} {scores[pos]:<9.4f} {t:<50} {g}")
                print()

    clear_memory(empty_cuda=False)
    print("Try other queries: 'The Matrix', 'Parasite', 'Pulp Fiction', 'Spirited Away'")

---

## Run Report

After the full run completes, the following artifacts are available:

| File | Location | Description |
|---|---|---|
| `tmdb_qwen06b_1024d.h5` | `/kaggle/working/` | Merged embeddings HDF5 |
| `trekomend_v2_1024d.zip` | `/kaggle/working/` | Downloadable zip (HDF5 + manifest + log) |
| `manifest.json` | `trekomend_v2_output/` | Full run metadata |
| `run.log` | `trekomend_v2_output/` | Detailed session log |

### Download

- **Kaggle Notebook Output tab**: Click the three dots \u22ee  Download
- **IPython FileLink**: Click the link in Cell 13 output
- **Kaggle Dataset**: Save as a Kaggle dataset for reuse in other kernels

### Verify Quality

```python
import h5py, numpy as np
with h5py.File("tmdb_qwen06b_1024d.h5", "r") as f:
    emb = f["embeddings"][:]
    print(f"Shape: {emb.shape}")
    print(f"Norm mean: {np.linalg.norm(emb, axis=1).mean():.4f}")  # Should be ~1.0
    print(f"No NaN: {np.isfinite(emb).all()}")                      # Should be True
```

---

## Troubleshooting

| Issue | Cause | Fix |
|---|---|---|
| Only 1 GPU detected | Settings wrong | Kaggle  Settings  **GPU T4 x2**  toggle off/on |
| `torch.cuda.OutOfMemoryError` | Batch too large | Auto-reduces. If persists, lower `TOTAL_BATCH_SIZE` in Cell 2 to 256 |
| `transformers < 4.51.0` | Old Kaggle image | Cell 1 upgrades automatically |
| Out of disk on `/kaggle/working` | Too many files persisted | Shards auto-delete after merge. Delete unneeded zips: `!rm /kaggle/working/*.zip` |
| Speed < 200 rows/s | Single GPU or eager attn | Check Cell 7 output. Verify dual GPU in Cell 4. Confirm flash_attention_2. |
| Kernel dies (OOM) | RAM exhaustion | Reduce `SHARD_SIZE` to 5_000 in Cell 2 |
| flash-attn install fails | Old CUDA/PyTorch | Falls back to sdpa automatically. No action needed. |
| torch.compile fails | PyTorch < 2.0 or unsupported op | Falls back to eager. No action needed. |
| Dataset not found | Missing from kernel metadata | Run Cell 4  auto-downloads from HuggingFace as fallback |

### Expected Output Sizes (~1M movies)

| File | 1024-dim | 768-dim | 512-dim |
|---|---|---|---|
| Per shard (10K rows) | ~40 MB | ~30 MB | ~20 MB |
| All shards (~100) | ~3.9 GB | ~2.9 GB | ~2.0 GB |
| Merged HDF5 (gzip lvl 2) | ~3.8 GB | ~2.8 GB | ~1.9 GB |
| Zip (stored) | ~3.8 GB | ~2.8 GB | ~1.9 GB |

### Dimension Tradeoff

Change `OUT_DIM` in Cell 2 to balance quality vs file size:
- **1024**: Full native quality, ~3.8 GB  recommend for production
- **768**: ~2-3% quality loss, ~2.8 GB  good for limited storage
- **512**: ~5-8% quality loss, ~1.9 GB  mobile/edge deployment
- **256**: ~15-20% quality loss, ~1.0 GB  extreme storage constraint

### Model Switch

To use the **4B model** instead (higher quality, slower):
1. Change `MODEL_ID` in Cell 2 to `"Qwen/Qwen3-Embedding-4B"`
2. Reduce `TOTAL_BATCH_SIZE` to 64 (4B model needs more VRAM)
3. Increase `OUT_DIM` to 1536
4. Expect 3-4\u00d7 slower (but MTEB ~75 vs ~62 for 0.6B)